# Ternary Edge-RV: QAT Exploration Notebook

This notebook explores Quantization-Aware Training with Larq's STE ternary quantizer.
Use it to experiment with layer sizes, learning rates, and observe weight distributions.

In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import numpy as np
import tensorflow as tf
import larq as lq
import matplotlib.pyplot as plt

In [ ]:
# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784).astype("float32") / 255.0
x_test  = x_test.reshape(-1, 784).astype("float32") / 255.0

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

In [ ]:
def build_ternary_mlp(hidden1=1024, hidden2=512, hidden3=256):
    return tf.keras.models.Sequential([
        lq.layers.QuantDense(hidden1, input_shape=(784,), use_bias=False,
                             kernel_quantizer="ste_tern", kernel_constraint="weight_clip"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        lq.layers.QuantDense(hidden2, use_bias=False,
                             kernel_quantizer="ste_tern", kernel_constraint="weight_clip"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        lq.layers.QuantDense(hidden3, use_bias=False,
                             kernel_quantizer="ste_tern", kernel_constraint="weight_clip"),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.Dense(10, activation="softmax"),
    ])

model = build_ternary_mlp()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(x_train, y_train, epochs=20, batch_size=256,
                    validation_split=0.1, verbose=1)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["loss"], label="Train")
ax1.plot(history.history["val_loss"], label="Val")
ax1.set_title("Loss"); ax1.legend(); ax1.set_xlabel("Epoch")
ax2.plot(history.history["accuracy"], label="Train")
ax2.plot(history.history["val_accuracy"], label="Val")
ax2.set_title("Accuracy"); ax2.legend(); ax2.set_xlabel("Epoch")
plt.show()

In [ ]:
# Inspect weight distributions
for layer in model.layers:
    if isinstance(layer, lq.layers.QuantDense):
        w = layer.get_weights()[0]
        unique, counts = np.unique(np.round(w, decimals=4), return_counts=True)
        plt.bar([str(v) for v in unique], counts)
        plt.title(f"{layer.name}")
        plt.xlabel("Weight Value"); plt.ylabel("Count")
        plt.show()

In [ ]:
# Evaluate and save
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {acc*100:.2f}%")
model.save("ternary_mnist_qat.h5")
print("Model saved.")